# Example A: Gradient Verification with the Taylor Test

This notebook verifies that JAX's automatic differentiation (AD) produces **correct gradients**
through the Shallow Water Model time integration.

## Background

The **Taylor test** is the standard way to verify AD-computed gradients. For a differentiable
scalar function $J(m)$ with gradient $\nabla J$:

- **First-order remainder** (no gradient): $|J(m + h\,\delta m) - J(m)| = O(h)$
- **Second-order remainder** (using gradient): $|J(m + h\,\delta m) - J(m) - h\,\nabla J \cdot \delta m| = O(h^2)$

If the second-order remainder converges at rate 2 (i.e., halving $h$ quarters the remainder),
the gradient is correct.

## Outline

1. Set up the SWM as a pure JAX function
2. Define a scalar cost function on the final state
3. Compute gradients with `jax.grad`
4. Run the Taylor test and verify second-order convergence
5. Compare `jax.lax.scan` vs Python for-loop approaches

## 1. Imports and Setup

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

# Enable double precision — essential for clean Taylor test convergence
jax.config.update("jax_enable_x64", True)

# Import the SWM functions
from swm_array_api import initialize_interior, _interior_to_halo, timestep

print(f"JAX version: {jax.__version__}")
print(f"Default device: {jax.devices()[0]}")
print(f"Float64 enabled: {jax.x64_is_enabled()}")

## 2. Model Parameters

We use a small grid and short integration to keep gradients well-behaved.
The leapfrog scheme with Asselin filter is stable for these parameters.

In [ ]:
# Grid and physics
M, N = 16, 16
dx = 100000.0
dy = 100000.0
dt = 90.0
a = 1000000.0
alpha = 0.001

# Number of timesteps for the AD integration
# We start with a modest number; longer integrations can cause
# gradient issues (the "chaotic adjoint" problem).
N_STEPS = 100

print(f"Grid: {M} x {N}")
print(f"Timesteps: {N_STEPS}")
print(f"Physical time: {N_STEPS * dt:.0f} s = {N_STEPS * dt / 3600:.1f} hours")

## 3. Wrapping the SWM as a Pure JAX Function

The existing `timestep` function in `swm_array_api.py` is already written in a
functional style (no in-place mutations), which makes it directly compatible with JAX AD.

We need to:
1. Pack the state into a single structure (tuple of arrays) for `jax.lax.scan`
2. Handle the first-step special case (forward Euler with `tdt = dt`, `alpha = 0`)
   vs. subsequent steps (leapfrog with `tdt = 2*dt`, `alpha = 0.001`)

In [ ]:
xp = jnp  # Use jax.numpy as our array namespace


def initial_state(u_interior, v_interior, p_interior):
    """Build the full initial state (with halos) from interior fields.

    Returns (u, v, p, uold, vold, pold) — for leapfrog,
    the 'old' fields start as copies of the current fields.
    """
    u = _interior_to_halo(xp, u_interior)
    v = _interior_to_halo(xp, v_interior)
    p = _interior_to_halo(xp, p_interior)
    return (u, v, p, u, v, p)  # old = current at t=0


def forward_model_scan(u_int, v_int, p_int, n_steps):
    """Run the SWM forward using jax.lax.scan (efficient for AD).

    Args:
        u_int, v_int, p_int: Initial interior fields, shape (M, N).
        n_steps: Number of timesteps.

    Returns:
        Final state (u, v, p) with halos, shape (M+2, N+2).
    """
    state = initial_state(u_int, v_int, p_int)

    # First step: forward Euler (tdt=dt, alpha=0)
    u, v, p, uold, vold, pold = state
    unew, vnew, pnew, uold, vold, pold = timestep(
        xp, u, v, p, uold, vold, pold, dx, dy, dt, 0.0, M, N
    )

    # Subsequent steps via scan: leapfrog (tdt=2*dt, alpha=alpha)
    carry = (unew, vnew, pnew, uold, vold, pold)

    def scan_step(carry, _):
        u, v, p, uold, vold, pold = carry
        unew, vnew, pnew, uold_new, vold_new, pold_new = timestep(
            xp, u, v, p, uold, vold, pold, dx, dy, 2.0 * dt, alpha, M, N
        )
        return (unew, vnew, pnew, uold_new, vold_new, pold_new), None

    final_carry, _ = jax.lax.scan(scan_step, carry, None, length=n_steps - 1)
    u_final, v_final, p_final = final_carry[0], final_carry[1], final_carry[2]
    return u_final, v_final, p_final


def forward_model_loop(u_int, v_int, p_int, n_steps):
    """Run the SWM forward using a Python for-loop.

    Simpler to read, but creates a large computation graph (one node per step).
    Fine for short integrations; use scan for longer ones.
    """
    state = initial_state(u_int, v_int, p_int)
    u, v, p, uold, vold, pold = state

    for ncycle in range(n_steps):
        tdt = dt if ncycle == 0 else 2.0 * dt
        alpha_val = 0.0 if ncycle == 0 else alpha
        unew, vnew, pnew, uold, vold, pold = timestep(
            xp, u, v, p, uold, vold, pold, dx, dy, tdt, alpha_val, M, N
        )
        u, v, p = unew, vnew, pnew

    return u, v, p


print("Forward model functions defined.")

## 4. Define the Cost Function

We define a scalar cost function on the final state:

$$J(u_0, v_0, p_0) = \frac{1}{2} \sum_{i,j} \left[ (p^{\text{final}}_{i,j} - p^{\text{target}}_{i,j})^2 + (u^{\text{final}}_{i,j} - u^{\text{target}}_{i,j})^2 + (v^{\text{final}}_{i,j} - v^{\text{target}}_{i,j})^2 \right]
$$

where the target is simply the forward model run from the default initial condition.
We differentiate $J$ with respect to the initial interior fields $(u_0, v_0, p_0)$.

In [ ]:
# Generate the default initial condition
u0_int, v0_int, p0_int = initialize_interior(xp, M, N, dx, dy, a)

# Run the forward model to get the "target" final state
print("Running forward model to compute target state...")
u_target, v_target, p_target = forward_model_scan(u0_int, v0_int, p0_int, N_STEPS)
# Block until computation is done
p_target.block_until_ready()
print(f"Target state computed. p range: [{float(p_target.min()):.2f}, {float(p_target.max()):.2f}]")


def cost_function(u_int, v_int, p_int, forward_fn=forward_model_scan):
    """Scalar cost: half the squared L2 distance from final state to target."""
    u_final, v_final, p_final = forward_fn(u_int, v_int, p_int, N_STEPS)
    # Sum over interior only (avoid double-counting halos)
    du = u_final[1:-1, 1:-1] - u_target[1:-1, 1:-1]
    dv = v_final[1:-1, 1:-1] - v_target[1:-1, 1:-1]
    dp = p_final[1:-1, 1:-1] - p_target[1:-1, 1:-1]
    return 0.5 * jnp.sum(du**2 + dv**2 + dp**2)


# Verify: cost at the true initial condition should be ~0
J0 = cost_function(u0_int, v0_int, p0_int)
print(f"Cost at true initial condition: {float(J0):.2e} (should be ~0)")

## 5. Compute Gradients with JAX

`jax.grad` computes the gradient of a scalar function with respect to its arguments.
We use `argnums=(0, 1, 2)` to differentiate w.r.t. all three initial fields.

In [ ]:
# Create the gradient function
# jax.value_and_grad returns both the cost value and the gradient
cost_and_grad = jax.value_and_grad(cost_function, argnums=(0, 1, 2))

# Evaluate at a perturbed initial condition (not the exact target)
# to get a nonzero cost and gradient
key = jax.random.PRNGKey(42)
keys = jax.random.split(key, 3)

# Scale perturbation relative to field magnitudes
u_scale = float(jnp.std(u0_int))
v_scale = float(jnp.std(v0_int))
p_scale = float(jnp.std(p0_int))

perturbation_size = 0.01  # 1% perturbation
u_pert = u0_int + perturbation_size * u_scale * jax.random.normal(keys[0], u0_int.shape)
v_pert = v0_int + perturbation_size * v_scale * jax.random.normal(keys[1], v0_int.shape)
p_pert = p0_int + perturbation_size * p_scale * jax.random.normal(keys[2], p0_int.shape)

print("Computing cost and gradient at perturbed state...")
J_val, (grad_u, grad_v, grad_p) = cost_and_grad(u_pert, v_pert, p_pert)
grad_p.block_until_ready()

print(f"Cost at perturbed state: {float(J_val):.6e}")
print(f"Gradient norms:")
print(f"  |∇_u J| = {float(jnp.linalg.norm(grad_u)):.6e}")
print(f"  |∇_v J| = {float(jnp.linalg.norm(grad_v)):.6e}")
print(f"  |∇_p J| = {float(jnp.linalg.norm(grad_p)):.6e}")

## 6. The Taylor Test

We verify the gradient by checking the convergence rate of:

- **$r_1(h) = |J(m + h\,\delta m) - J(m)|$** — should be $O(h)$, i.e., ratio ~2 when halving $h$
- **$r_2(h) = |J(m + h\,\delta m) - J(m) - h\,\nabla J \cdot \delta m|$** — should be $O(h^2)$, i.e., ratio ~4

A convergence rate of 2 for $r_2$ **proves** the gradient is correct.

In [ ]:
def taylor_test(cost_fn, grad_tuple, m_tuple, dm_tuple, h_init=1e-3, n_steps=8):
    """Run the Taylor test for a cost function with multiple input arrays.

    Args:
        cost_fn: Scalar function of (u_int, v_int, p_int).
        grad_tuple: (grad_u, grad_v, grad_p) at the base point.
        m_tuple: (u_int, v_int, p_int) base point.
        dm_tuple: (du, dv, dp) perturbation direction.
        h_init: Initial step size.
        n_steps: Number of refinement steps.

    Returns:
        h_vals, r1_vals, r2_vals: Arrays of step sizes and remainders.
    """
    J0 = cost_fn(*m_tuple)

    # Directional derivative: sum of grad_i · dm_i
    directional_deriv = sum(
        jnp.sum(g * d) for g, d in zip(grad_tuple, dm_tuple)
    )

    h_vals = []
    r1_vals = []
    r2_vals = []

    h = h_init
    for i in range(n_steps):
        # Perturbed point: m + h * dm
        m_h = tuple(mi + h * dmi for mi, dmi in zip(m_tuple, dm_tuple))
        J_h = cost_fn(*m_h)

        r1 = abs(float(J_h - J0))
        r2 = abs(float(J_h - J0 - h * directional_deriv))

        h_vals.append(h)
        r1_vals.append(r1)
        r2_vals.append(r2)

        h /= 2.0

    return np.array(h_vals), np.array(r1_vals), np.array(r2_vals)


# Random perturbation direction (normalized)
key = jax.random.PRNGKey(123)
keys = jax.random.split(key, 3)
dm_u = jax.random.normal(keys[0], u_pert.shape)
dm_v = jax.random.normal(keys[1], v_pert.shape)
dm_p = jax.random.normal(keys[2], p_pert.shape)

# Normalize so the perturbation direction has unit norm
dm_norm = jnp.sqrt(jnp.sum(dm_u**2) + jnp.sum(dm_v**2) + jnp.sum(dm_p**2))
dm_u = dm_u / dm_norm
dm_v = dm_v / dm_norm
dm_p = dm_p / dm_norm

print("Running Taylor test...")
h_vals, r1_vals, r2_vals = taylor_test(
    lambda u, v, p: cost_function(u, v, p),
    (grad_u, grad_v, grad_p),
    (u_pert, v_pert, p_pert),
    (dm_u, dm_v, dm_p),
    h_init=1e-2,
    n_steps=10,
)

# Print results
print(f"\n{'h':>12s}  {'|r1|':>14s}  {'|r2|':>14s}  {'r1 ratio':>10s}  {'r2 ratio':>10s}")
print("-" * 68)
for i in range(len(h_vals)):
    r1_ratio = r1_vals[i - 1] / r1_vals[i] if i > 0 and r1_vals[i] > 0 else float('nan')
    r2_ratio = r2_vals[i - 1] / r2_vals[i] if i > 0 and r2_vals[i] > 0 else float('nan')
    print(f"{h_vals[i]:12.2e}  {r1_vals[i]:14.6e}  {r2_vals[i]:14.6e}  {r1_ratio:10.4f}  {r2_ratio:10.4f}")

print(f"\nExpected: r1 ratio ≈ 2.0 (first-order), r2 ratio ≈ 4.0 (second-order)")
print(f"If r2 ratio ≈ 4.0, the gradient is CORRECT.")

## 7. Visualize the Taylor Test Results

A log-log plot is the standard way to display the Taylor test.
The slopes should be ~1 for $r_1$ and ~2 for $r_2$.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

ax.loglog(h_vals, r1_vals, 'bo-', label=r'$r_1 = |J(m+h\delta m) - J(m)|$ (no gradient)', linewidth=2)
ax.loglog(h_vals, r2_vals, 'rs-', label=r'$r_2 = |J(m+h\delta m) - J(m) - h\nabla J \cdot \delta m|$ (with gradient)', linewidth=2)

# Reference slopes
h_ref = np.array([h_vals[0], h_vals[-1]])
# O(h) reference line
r1_ref = r1_vals[0] * (h_ref / h_vals[0])
ax.loglog(h_ref, r1_ref, 'b--', alpha=0.4, label='O(h) reference (slope 1)')
# O(h^2) reference line
r2_ref = r2_vals[0] * (h_ref / h_vals[0])**2
ax.loglog(h_ref, r2_ref, 'r--', alpha=0.4, label='O(h²) reference (slope 2)')

ax.set_xlabel('Step size h', fontsize=13)
ax.set_ylabel('Remainder', fontsize=13)
ax.set_title('Taylor Test: Gradient Verification for SWM (JAX AD)', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Visualize the Gradient Fields

Let's look at the gradient $\nabla_p J$ — it tells us: "how does changing each grid
point of the initial pressure field affect the cost?"

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

fields = [grad_u, grad_v, grad_p]
names = [r'$\nabla_u J$', r'$\nabla_v J$', r'$\nabla_p J$']

for ax, field, name in zip(axes, fields, names):
    fdata = np.array(field)
    vmax = np.abs(fdata).max()
    im = ax.imshow(fdata, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
    ax.set_title(name, fontsize=14)
    plt.colorbar(im, ax=ax, shrink=0.8)

fig.suptitle(f'Gradient of cost function w.r.t. initial conditions (N_STEPS={N_STEPS})', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 9. Comparison: `jax.lax.scan` vs Python For-Loop

Both approaches should give identical gradients, but they differ in:
- **Compilation**: `scan` compiles a single step body; the Python loop unrolls the full graph
- **Memory**: `scan` can be more memory-efficient with checkpointing
- **Compile time**: `scan` is faster to compile; the for-loop compile time grows with `n_steps`

In [ ]:
from time import perf_counter

# --- Scan-based gradient ---
print("=== jax.lax.scan approach ===")

cost_scan = lambda u, v, p: cost_function(u, v, p, forward_fn=forward_model_scan)
grad_fn_scan = jax.value_and_grad(cost_scan, argnums=(0, 1, 2))

# First call includes JIT compilation
t0 = perf_counter()
J_scan, grads_scan = grad_fn_scan(u_pert, v_pert, p_pert)
grads_scan[2].block_until_ready()
t_compile_scan = perf_counter() - t0
print(f"  First call (includes compile): {t_compile_scan:.3f}s")

# Second call is pure execution
t0 = perf_counter()
J_scan, grads_scan = grad_fn_scan(u_pert, v_pert, p_pert)
grads_scan[2].block_until_ready()
t_exec_scan = perf_counter() - t0
print(f"  Second call (cached):          {t_exec_scan:.3f}s")

# --- Python for-loop gradient ---
print(f"\n=== Python for-loop approach (N_STEPS={N_STEPS}) ===")

cost_loop = lambda u, v, p: cost_function(u, v, p, forward_fn=forward_model_loop)
grad_fn_loop = jax.jit(jax.value_and_grad(cost_loop, argnums=(0, 1, 2)))

t0 = perf_counter()
J_loop, grads_loop = grad_fn_loop(u_pert, v_pert, p_pert)
grads_loop[2].block_until_ready()
t_compile_loop = perf_counter() - t0
print(f"  First call (includes compile): {t_compile_loop:.3f}s")

t0 = perf_counter()
J_loop, grads_loop = grad_fn_loop(u_pert, v_pert, p_pert)
grads_loop[2].block_until_ready()
t_exec_loop = perf_counter() - t0
print(f"  Second call (cached):          {t_exec_loop:.3f}s")

# Verify they agree
print(f"\n=== Agreement ===")
print(f"  Cost difference: {abs(float(J_scan) - float(J_loop)):.2e}")
print(f"  Grad_u max diff: {float(jnp.max(jnp.abs(grads_scan[0] - grads_loop[0]))):.2e}")
print(f"  Grad_v max diff: {float(jnp.max(jnp.abs(grads_scan[1] - grads_loop[1]))):.2e}")
print(f"  Grad_p max diff: {float(jnp.max(jnp.abs(grads_scan[2] - grads_loop[2]))):.2e}")

## 10. Effect of Integration Length on Gradients

Longer integrations can lead to the "chaotic adjoint" problem: gradients grow
exponentially or oscillate wildly. Let's see how the gradient norm evolves
with the number of timesteps.

In [ ]:
step_counts = [10, 25, 50, 100, 200, 500]
grad_norms = []

print("Computing gradient norms for different integration lengths...")
for ns in step_counts:
    def cost_ns(u, v, p, _ns=ns):
        u_f, v_f, p_f = forward_model_scan(u, v, p, _ns)
        du = u_f[1:-1, 1:-1] - u_target[1:-1, 1:-1]
        dv = v_f[1:-1, 1:-1] - v_target[1:-1, 1:-1]
        dp = p_f[1:-1, 1:-1] - p_target[1:-1, 1:-1]
        return 0.5 * jnp.sum(du**2 + dv**2 + dp**2)

    val, grads = jax.value_and_grad(cost_ns, argnums=(0, 1, 2))(u_pert, v_pert, p_pert)
    grads[2].block_until_ready()
    gnorm = float(jnp.sqrt(sum(jnp.sum(g**2) for g in grads)))
    grad_norms.append(gnorm)
    print(f"  N_STEPS={ns:4d}  |cost|={float(val):.4e}  |∇J|={gnorm:.4e}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogy(step_counts, grad_norms, 'ko-', linewidth=2, markersize=8)
ax.set_xlabel('Number of timesteps', fontsize=13)
ax.set_ylabel('Gradient norm |∇J|', fontsize=13)
ax.set_title('Gradient norm vs. integration length', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

**Key takeaways:**

1. The `swm_array_api.py` implementation is **directly differentiable** with JAX — no code changes needed.
   The functional, mutation-free style pays off: `jax.grad` just works.

2. The **Taylor test confirms** second-order convergence ($r_2$ ratio $\approx 4$), proving
   the AD-computed gradient is correct.

3. Both `jax.lax.scan` and Python for-loop approaches give **identical gradients**,
   but `scan` compiles faster and scales better to long integrations.

4. For moderate integration lengths, gradients are well-behaved. Very long integrations
   may exhibit gradient growth (the chaotic adjoint problem).

This verified gradient is the foundation for **Example B** (4D-Var data assimilation),
where we use the gradient to optimize initial conditions.